# Survival Prediction & Race as Explicit Feature


### Experiment 1: Survival rate prediction
Flip the outcome label: predict who SURVIVES instead of who DIES.


### Experiment 2: Race as explicit feature
Train all four models WITH race as an explicit one-hot input feature.
Compare fairness metrics directly against the baseline (no race feature)
and adversarial debiasing results.

## Key comparison table:
- Baseline (no race feature)
- With race feature (explicit)
- Adversarial debiasing (lambda=3, best tradeoff)

## Fairness metrics used:
- Equal Opportunity gap (TPR)
- Equalized Odds gap (TPR + FPR)
- Balanced Error Rate gap

---
## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from pathlib import Path
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_curve, roc_curve
)
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

OUT_DIR  = Path('/home/ino/thesis_outputs')
EXP_DIR  = OUT_DIR / 'survival_race_feature'
EXP_DIR.mkdir(exist_ok=True)

VITAL_NAMES  = ['heart_rate', 'resp_rate', 'spo2', 'temperature', 'map']
KEEP_RACES   = ['White', 'Black', 'Hispanic', 'Asian']
RACE_TO_INT  = {g: i for i, g in enumerate(KEEP_RACES)}
N_FOLDS      = 5
RANDOM_STATE = 42
N_BOOTSTRAP  = 2000
THRESHOLD    = 0.5

# Training hyperparameters
N_EPOCHS_LSTM  = 50
N_EPOCHS_TRANS = 60
N_EPOCHS_DNN   = 80
BATCH_SIZE     = 512
LR_LSTM        = 5e-4
LR_TRANS       = 3e-4
LR_DNN         = 3e-4

COLORS = {
    'XGBoost':     'steelblue',
    'Deep NN':     'seagreen',
    'LSTM':        'darkorange',
    'Transformer': 'mediumpurple',
}
RACE_COLORS = {
    'White': '#4878CF', 'Black': '#D65F5F',
    'Hispanic': '#6ACC65', 'Asian': '#B47CC7'
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpus = torch.cuda.device_count()
print(f'Device: {device}  |  GPUs: {n_gpus}')
print(f'Output: {EXP_DIR}')

---
## 2. Load & align data

In [ ]:
cv_stay_ids = np.load(OUT_DIR / 'cv_stay_ids.npy')
print(f'CV stay_ids: {len(cv_stay_ids):,}')

cohort_full = pd.read_csv(OUT_DIR / 'cohort.csv')
cohort_full = cohort_full.set_index('stay_id')
cohort_cv   = cohort_full.loc[cv_stay_ids].reset_index()

cohort_full_reset = pd.read_csv(OUT_DIR / 'cohort.csv').reset_index(drop=True)
stay_id_to_ts_idx = {sid: i for i, sid in
                     enumerate(cohort_full_reset['stay_id'].values)}

X_ts_full  = np.load(OUT_DIR / 'X_ts.npy')
M_ts_full  = np.load(OUT_DIR / 'M_ts.npy')
ts_indices = np.array([stay_id_to_ts_idx[sid] for sid in cv_stay_ids])
X_ts_cv    = X_ts_full[ts_indices]
M_ts_cv    = M_ts_full[ts_indices]

keep_mask = cohort_cv['race_group'].isin(KEEP_RACES).values
cohort    = cohort_cv[keep_mask].reset_index(drop=True)
kept_idx  = np.where(keep_mask)[0]

X_ts    = X_ts_cv[kept_idx]
M_ts    = M_ts_cv[kept_idx]

# Build tabular features
print('Building tabular features...')
events = pd.read_parquet(OUT_DIR / 'events_24h.parquet')
agg = events.groupby(['stay_id','vital'])['valuenum'].agg(
    ['mean','std','min','max']).unstack('vital')
agg.columns = [f'{vital}_{stat}' for stat, vital in agg.columns]
agg = agg.reset_index()

df_tab = cohort[['stay_id','hospital_expire_flag','anchor_age',
                 'race_group']].merge(agg, on='stay_id', how='inner')

tab_mask  = cohort['stay_id'].isin(df_tab['stay_id'].values).values
cohort_f  = cohort[tab_mask].reset_index(drop=True)
X_ts_f    = X_ts[tab_mask]
M_ts_f    = M_ts[tab_mask]
race_arr  = cohort_f['race_group'].values
race_int  = np.array([RACE_TO_INT[r] for r in race_arr])

# One-hot encode race for explicit feature experiment
race_onehot = np.zeros((len(cohort_f), len(KEEP_RACES)), dtype=np.float32)
for i, r in enumerate(race_arr):
    race_onehot[i, RACE_TO_INT[r]] = 1.0

feature_cols = [c for c in df_tab.columns
                if c not in ['stay_id','hospital_expire_flag','race_group']]
X_tab = df_tab[feature_cols].values.astype(np.float64)

# Mortality labels
y_mort = cohort_f['hospital_expire_flag'].values.astype(np.int32)
# Survival labels — flipped
y_surv = (1 - y_mort).astype(np.int32)

print(f'Cohort: {len(cohort_f):,}')
print(f'Mortality rate: {y_mort.mean():.1%}')
print(f'Survival rate:  {y_surv.mean():.1%}')
print(pd.Series(race_arr).value_counts().to_string())

# Load existing baseline OOF probs for comparison
oof_baseline = {}
for m, fname in [('XGBoost','oof_probs_XGBoost.npy'),
                 ('Deep NN','oof_probs_Deep_NN.npy'),
                 ('LSTM','oof_probs_LSTM.npy')]:
    full = np.load(OUT_DIR / fname)[kept_idx]
    oof_baseline[m] = full[tab_mask]
trans_path = OUT_DIR / 'oof_probs_Transformer.npy'
if trans_path.exists():
    try:
        t = np.load(trans_path)
        if len(t) >= kept_idx.max()+1:
            oof_baseline['Transformer'] = t[kept_idx][tab_mask]
    except Exception:
        pass

print('\nBaseline AUPRC (mortality):')
for m, probs in oof_baseline.items():
    print(f'  {m:<14} {average_precision_score(y_mort, probs):.4f}')

---
## 3. Fairness metrics & fold splits

In [ ]:
def compute_fairness_metrics(y_true, probs, race, threshold=THRESHOLD):
    results = {}
    for g in ['Black', 'White']:
        mask = race == g
        yg, pg = y_true[mask], probs[mask]
        pred   = (pg >= threshold).astype(int)
        tp = ((pred==1)&(yg==1)).sum()
        fn = ((pred==0)&(yg==1)).sum()
        fp = ((pred==1)&(yg==0)).sum()
        tn = ((pred==0)&(yg==0)).sum()
        tpr = tp/(tp+fn) if (tp+fn)>0 else 0.0
        fpr = fp/(fp+tn) if (fp+tn)>0 else 0.0
        fnr = fn/(tp+fn) if (tp+fn)>0 else 0.0
        results[g] = {
            'auprc': average_precision_score(yg, pg),
            'auroc': roc_auc_score(yg, pg),
            'tpr': tpr, 'fpr': fpr, 'fnr': fnr,
            'ber': (fpr+fnr)/2,
        }
    gaps = {
        'auprc_gap':         results['White']['auprc'] - results['Black']['auprc'],
        'equal_opportunity': abs(results['Black']['tpr'] - results['White']['tpr']),
        'equalized_odds':    max(
            abs(results['Black']['tpr'] - results['White']['tpr']),
            abs(results['Black']['fpr'] - results['White']['fpr'])),
        'ber_gap':           abs(results['Black']['ber'] - results['White']['ber']),
    }
    return results, gaps


# Fold splits
strat_label = (pd.Series(race_arr).astype(str) + '_' +
               pd.Series(y_mort).astype(str)).values
skf         = StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                              random_state=RANDOM_STATE)
fold_splits  = list(skf.split(np.zeros(len(y_mort)), strat_label))
print(f'Fold test sizes: {[len(t) for _,t in fold_splits]}')

# Sanity check baseline fairness metrics
print('\nBaseline LSTM fairness metrics (mortality):')
if 'LSTM' in oof_baseline:
    _, gaps = compute_fairness_metrics(y_mort, oof_baseline['LSTM'], race_arr)
    for k,v in gaps.items():
        print(f'  {k}: {v:.4f}')

---
## 4. Model definitions

In [ ]:
class MortalityLSTM(nn.Module):
    def __init__(self, input_size=10, hidden_size=128,
                 num_layers=3, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout, bidirectional=True)
        self.layer_norm = nn.LayerNorm(hidden_size*2)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size*2, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64),            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1))
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.classifier(
            self.dropout(self.layer_norm(out[:,-1,:]))).squeeze(1)


class MortalityTransformer(nn.Module):
    def __init__(self, input_size=10, d_model=128, nhead=8,
                 num_layers=3, dropout=0.3, max_len=24):
        super().__init__()
        self.input_proj    = nn.Linear(input_size, d_model)
        self.pos_embedding = nn.Embedding(max_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model*4, dropout=dropout,
            batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers,
            enable_nested_tensor=False)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        pos = torch.arange(x.shape[1], device=x.device)
        x   = self.input_proj(x) + self.pos_embedding(pos)
        return self.classifier(
            self.transformer(self.dropout(x)).mean(dim=1)).squeeze(1)


class MortalityDNN(nn.Module):
    def __init__(self, input_size=240, dropout=0.3):
        super().__init__()
        self.b1 = nn.Sequential(
            nn.Linear(input_size,512), nn.BatchNorm1d(512),
            nn.ReLU(), nn.Dropout(dropout))
        self.b2 = nn.Sequential(
            nn.Linear(512,256), nn.BatchNorm1d(256),
            nn.ReLU(), nn.Dropout(dropout))
        self.b3 = nn.Sequential(
            nn.Linear(256,128), nn.BatchNorm1d(128),
            nn.ReLU(), nn.Dropout(dropout))
        self.b4 = nn.Sequential(
            nn.Linear(128,64), nn.BatchNorm1d(64),
            nn.ReLU(), nn.Dropout(dropout))
        self.res = nn.Linear(256, 64)
        self.out = nn.Linear(64, 1)
    def forward(self, x):
        x  = x.view(x.size(0), -1)
        x  = self.b1(x)
        x2 = self.b2(x)
        x  = self.b4(self.b3(x2)) + self.res(x2)
        return self.out(x).squeeze(1)


def train_nn(model, train_loader, val_loader, y_te,
             n_epochs, lr, use_cosine=False, label=''):
    pw = torch.tensor(
        [(y_te==0).sum()/(y_te==1).sum()],
        dtype=torch.float32).to(device)
    crit = nn.BCEWithLogitsLoss(pos_weight=pw)
    opt  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    if use_cosine:
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt, T_max=n_epochs, eta_min=1e-5)
    else:
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode='max', patience=3, factor=0.5)

    best_ap, best_p = 0, None
    for ep in range(1, n_epochs+1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        if use_cosine: sched.step()
        model.eval()
        pl = []
        with torch.no_grad():
            for xb, _ in val_loader:
                pl.append(torch.sigmoid(model(xb.to(device))).cpu().numpy())
        p  = np.concatenate(pl)
        ap = average_precision_score(y_te, p)
        if not use_cosine: sched.step(ap)
        if ap > best_ap:
            best_ap, best_p = ap, p.copy()
        if ep % 10 == 0 or ep == 1:
            print(f'    [{label}] ep {ep:02d}  AUPRC={ap:.4f}'
                  f'{"  *" if ap==best_ap else ""}')
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return best_p, best_ap


def normalise(X, X_train):
    m = X_train.mean(axis=(0,1), keepdims=True)
    s = X_train.std(axis=(0,1),  keepdims=True) + 1e-8
    return (X - m) / s


def make_loaders(X_tr, y_tr, X_te, y_te):
    y_tr_t = torch.tensor(y_tr, dtype=torch.float32)
    y_te_t = torch.tensor(y_te, dtype=torch.float32)
    tr = DataLoader(TensorDataset(torch.tensor(X_tr), y_tr_t),
                    batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=4, pin_memory=True)
    te = DataLoader(TensorDataset(torch.tensor(X_te), y_te_t),
                    batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=4, pin_memory=True)
    return tr, te


print('All model classes defined.')

---
## 5. Experiment 1: Survival rate prediction

All four models retrained with y_survival = 1 - y_mortality.

In [ ]:
print('EXPERIMENT 1 — Survival rate prediction')
print('y = 1 means survived, y = 0 means died')
print(f'Survival prevalence: {y_surv.mean():.1%}')
print('=' * 65)

oof_surv = {m: np.zeros(len(y_mort))
            for m in ['XGBoost','Deep NN','LSTM','Transformer']}

for fold, (train_idx, test_idx) in enumerate(fold_splits):
    print(f'\n── Fold {fold+1}/{N_FOLDS} ')

    y_tr   = y_surv[train_idx]
    y_te   = y_surv[test_idx]
    scale_pw = (y_tr==0).sum() / (y_tr==1).sum()

    # XGBoost 
    imp = SimpleImputer(strategy='median')
    scl = StandardScaler()
    X_tr_tab = scl.fit_transform(imp.fit_transform(X_tab[train_idx]))
    X_te_tab = scl.transform(imp.transform(X_tab[test_idx]))
    xgb = XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pw, eval_metric='logloss',
        random_state=RANDOM_STATE, verbosity=0, device='cuda')
    xgb.fit(X_tr_tab, y_tr)
    oof_surv['XGBoost'][test_idx] = xgb.predict_proba(X_te_tab)[:,1]
    print(f'  XGBoost AUPRC={average_precision_score(y_te, oof_surv["XGBoost"][test_idx]):.4f}')

    # TS inputs
    X_tr_ts = X_ts_f[train_idx]
    X_te_ts = X_ts_f[test_idx]
    M_tr    = M_ts_f[train_idx]
    M_te    = M_ts_f[test_idx]
    X_tr_n  = normalise(X_tr_ts, X_tr_ts)
    X_te_n  = normalise(X_te_ts, X_tr_ts)
    X_tr_in = np.concatenate([X_tr_n, M_tr], axis=2).astype(np.float32)
    X_te_in = np.concatenate([X_te_n, M_te], axis=2).astype(np.float32)
    tr_ld, te_ld = make_loaders(X_tr_in, y_tr, X_te_in, y_te)

    # Deep NN   
    dnn = MortalityDNN(input_size=240).to(device)
    if n_gpus > 1: dnn = nn.DataParallel(dnn)
    p, _ = train_nn(dnn, tr_ld, te_ld, y_te,
                    N_EPOCHS_DNN, LR_DNN, use_cosine=True, label='DNN-surv')
    oof_surv['Deep NN'][test_idx] = p

    # LSTM
    lstm = MortalityLSTM().to(device)
    if n_gpus > 1: lstm = nn.DataParallel(lstm)
    p, _ = train_nn(lstm, tr_ld, te_ld, y_te,
                    N_EPOCHS_LSTM, LR_LSTM, use_cosine=False, label='LSTM-surv')
    oof_surv['LSTM'][test_idx] = p

    # Transformer 
    trans = MortalityTransformer().to(device)
    if n_gpus > 1: trans = nn.DataParallel(trans)
    p, _ = train_nn(trans, tr_ld, te_ld, y_te,
                    N_EPOCHS_TRANS, LR_TRANS, use_cosine=True, label='Trans-surv')
    oof_surv['Transformer'][test_idx] = p

for m, probs in oof_surv.items():
    np.save(EXP_DIR / f'oof_surv_{m.replace(" ","_")}.npy', probs)

print('\nExperiment 1 complete.')

In [ ]:
# Survival results
print('SURVIVAL PREDICTION RESULTS')
print('=' * 70)
print(f'{"Model":<14}  {"Overall":>8}  {"AUROC":>8}  '
      f'{"Black":>8}  {"White":>8}  {"Gap":>8}')
print('-' * 60)

surv_records = []
for m, probs in oof_surv.items():
    ov   = average_precision_score(y_surv, probs)
    ar   = roc_auc_score(y_surv, probs)
    bl   = average_precision_score(
        y_surv[race_arr=='Black'], probs[race_arr=='Black'])
    wh   = average_precision_score(
        y_surv[race_arr=='White'], probs[race_arr=='White'])
    gap  = wh - bl
    print(f'{m:<14}  {ov:>8.4f}  {ar:>8.4f}  {bl:>8.4f}  {wh:>8.4f}  {gap:>+8.4f}')
    _, gaps = compute_fairness_metrics(y_surv, probs, race_arr)
    surv_records.append({'model': m, 'overall': ov, 'auroc': ar,
                         'black': bl, 'white': wh, 'gap': gap, **gaps})

surv_df = pd.DataFrame(surv_records)
surv_df.to_csv(EXP_DIR / 'survival_results.csv', index=False)

print('\nSanity check. AUROC should match mortality AUROC:')
for m in oof_baseline:
    mort_ar = roc_auc_score(y_mort, oof_baseline[m])
    surv_ar = roc_auc_score(y_surv, oof_surv[m])
    print(f'  {m:<14}  mort AUROC={mort_ar:.4f}  surv AUROC={surv_ar:.4f}')

---
## 6. Experiment 2: Race as explicit feature

Race is one-hot encoded and appended to the inputs.
For XGBoost/DNN: appended to tabular feature vector.
For LSTM/Transformer: appended to each of the 24 time steps.
Input size grows from 10 to 14 for sequence models.

In [ ]:
print('EXPERIMENT 2 — Race as explicit feature')
print('Race one-hot appended to all model inputs')
print('=' * 65)

oof_race_feat = {m: np.zeros(len(y_mort))
                 for m in ['XGBoost','Deep NN','LSTM','Transformer']}

for fold, (train_idx, test_idx) in enumerate(fold_splits):
    print(f'\n── Fold {fold+1}/{N_FOLDS} ')

    y_tr     = y_mort[train_idx]
    y_te     = y_mort[test_idx]
    scale_pw = (y_tr==0).sum() / (y_tr==1).sum()

    # XGBoost with race appended to tabular features 
    imp = SimpleImputer(strategy='median')
    scl = StandardScaler()
    X_tr_tab = imp.fit_transform(X_tab[train_idx])
    X_te_tab = imp.transform(X_tab[test_idx])
    # Append race one-hot BEFORE scaling (race cols are already 0/1)
    X_tr_tab_r = np.hstack([X_tr_tab, race_onehot[train_idx]])
    X_te_tab_r = np.hstack([X_te_tab, race_onehot[test_idx]])
    X_tr_tab_r = scl.fit_transform(X_tr_tab_r)
    X_te_tab_r = scl.transform(X_te_tab_r)

    xgb = XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pw, eval_metric='logloss',
        random_state=RANDOM_STATE, verbosity=0, device='cuda')
    xgb.fit(X_tr_tab_r, y_tr)
    oof_race_feat['XGBoost'][test_idx] = xgb.predict_proba(X_te_tab_r)[:,1]
    print(f'  XGBoost AUPRC={average_precision_score(y_te, oof_race_feat["XGBoost"][test_idx]):.4f}')

    # TS inputs with race appended to each time step
    X_tr_ts = X_ts_f[train_idx]
    X_te_ts = X_ts_f[test_idx]
    M_tr    = M_ts_f[train_idx]
    M_te    = M_ts_f[test_idx]
    X_tr_n  = normalise(X_tr_ts, X_tr_ts)
    X_te_n  = normalise(X_te_ts, X_tr_ts)

    # Base input: (N, 24, 10) = vitals + mask
    X_tr_base = np.concatenate([X_tr_n, M_tr], axis=2)  # (N, 24, 10)
    X_te_base = np.concatenate([X_te_n, M_te], axis=2)

    # Append race one-hot to each time step: (N, 24, 14)
    race_tr_ts = np.tile(
        race_onehot[train_idx, np.newaxis, :],
        (1, 24, 1))  # (N, 24, 4)
    race_te_ts = np.tile(
        race_onehot[test_idx, np.newaxis, :],
        (1, 24, 1))

    X_tr_in = np.concatenate([X_tr_base, race_tr_ts], axis=2).astype(np.float32)
    X_te_in = np.concatenate([X_te_base, race_te_ts], axis=2).astype(np.float32)

    tr_ld, te_ld = make_loaders(X_tr_in, y_tr, X_te_in, y_te)

    # Deep NN (flatten 24x14=336) 
    dnn = MortalityDNN(input_size=24*14).to(device)
    if n_gpus > 1: dnn = nn.DataParallel(dnn)
    p, _ = train_nn(dnn, tr_ld, te_ld, y_te,
                    N_EPOCHS_DNN, LR_DNN, use_cosine=True, label='DNN+race')
    oof_race_feat['Deep NN'][test_idx] = p

    # LSTM (input_size=14) 
    lstm = MortalityLSTM(input_size=14).to(device)
    if n_gpus > 1: lstm = nn.DataParallel(lstm)
    p, _ = train_nn(lstm, tr_ld, te_ld, y_te,
                    N_EPOCHS_LSTM, LR_LSTM, use_cosine=False, label='LSTM+race')
    oof_race_feat['LSTM'][test_idx] = p

    # Transformer (input_size=14) 
    trans = MortalityTransformer(input_size=14).to(device)
    if n_gpus > 1: trans = nn.DataParallel(trans)
    p, _ = train_nn(trans, tr_ld, te_ld, y_te,
                    N_EPOCHS_TRANS, LR_TRANS, use_cosine=True, label='Trans+race')
    oof_race_feat['Transformer'][test_idx] = p

for m, probs in oof_race_feat.items():
    np.save(EXP_DIR / f'oof_race_feat_{m.replace(" ","_")}.npy', probs)

print('\nExperiment 2 complete.')

In [ ]:
# Race feature results
print('RACE AS EXPLICIT FEATURE RESULTS')
print('=' * 70)
print(f'{"Model":<14}  {"Overall":>8}  {"Black":>8}  {"White":>8}  {"Gap":>8}')
print('-' * 55)

race_feat_records = []
for m, probs in oof_race_feat.items():
    ov  = average_precision_score(y_mort, probs)
    bl  = average_precision_score(
        y_mort[race_arr=='Black'], probs[race_arr=='Black'])
    wh  = average_precision_score(
        y_mort[race_arr=='White'], probs[race_arr=='White'])
    gap = wh - bl
    print(f'{m:<14}  {ov:>8.4f}  {bl:>8.4f}  {wh:>8.4f}  {gap:>+8.4f}')
    _, gaps = compute_fairness_metrics(y_mort, probs, race_arr)
    race_feat_records.append({'model': m, 'overall': ov,
                              'black': bl, 'white': wh, 'gap': gap, **gaps})

race_feat_df = pd.DataFrame(race_feat_records)
race_feat_df.to_csv(EXP_DIR / 'race_feature_results.csv', index=False)

---
## 7. Figures

In [ ]:
# Figure 1: Survival vs Mortality AUPRC by race
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (y_use, probs_dict, label, baseline) in [
    (axes[0], (y_mort, oof_baseline, 'Mortality\n(baseline)', y_mort.mean())),
    (axes[1], (y_surv, oof_surv,     'Survival',              y_surv.mean())),
]:
    y_use, probs_dict, label, baseline = y_use, probs_dict, label, baseline
    models_plot = [m for m in ['XGBoost','Deep NN','LSTM','Transformer']
                   if m in probs_dict]
    x     = np.arange(len(KEEP_RACES))
    width = 0.2
    for offset, m in enumerate(models_plot):
        vals = [average_precision_score(
                    y_use[race_arr==g], probs_dict[m][race_arr==g])
                for g in KEEP_RACES]
        ax.bar(x + offset*width, vals, width, label=m,
               color=list(COLORS.values())[offset],
               edgecolor='white', alpha=0.85)
    ax.axhline(baseline, color='gray', linestyle='--', linewidth=1.2,
               label=f'Prevalence ({baseline:.2f})')
    ax.set_xticks(x + width*1.5)
    ax.set_xticklabels(KEEP_RACES)
    ax.set_ylabel('AUPRC')
    ax.set_title(f'{label} prediction\nAUPRC by race group')
    ax.legend(fontsize=8)

plt.suptitle('Survival vs Mortality prediction — racial disparities comparison',
             fontsize=13)
plt.tight_layout()
plt.savefig(EXP_DIR / 'fig_survival_vs_mortality.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_survival_vs_mortality.png')

In [ ]:
# Figure 2: Survival White-Black gap vs Mortality gap 
models_plot = [m for m in ['XGBoost','Deep NN','LSTM','Transformer']
               if m in oof_surv and m in oof_baseline]

mort_gaps = []
surv_gaps = []
for m in models_plot:
    wh_m = average_precision_score(
        y_mort[race_arr=='White'], oof_baseline[m][race_arr=='White'])
    bl_m = average_precision_score(
        y_mort[race_arr=='Black'], oof_baseline[m][race_arr=='Black'])
    wh_s = average_precision_score(
        y_surv[race_arr=='White'], oof_surv[m][race_arr=='White'])
    bl_s = average_precision_score(
        y_surv[race_arr=='Black'], oof_surv[m][race_arr=='Black'])
    mort_gaps.append(wh_m - bl_m)
    surv_gaps.append(wh_s - bl_s)

x     = np.arange(len(models_plot))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, mort_gaps, width, label='Mortality prediction',
       color='#D65F5F', edgecolor='white', alpha=0.85)
ax.bar(x + width/2, surv_gaps, width, label='Survival prediction',
       color='#4878CF', edgecolor='white', alpha=0.85)
for i, (mg, sg) in enumerate(zip(mort_gaps, surv_gaps)):
    ax.text(i-width/2, mg+0.001, f'{mg:.3f}', ha='center', fontsize=9)
    ax.text(i+width/2, sg+0.001, f'{sg:.3f}', ha='center', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(models_plot)
ax.set_ylabel('White AUPRC - Black AUPRC')
ax.set_title('Racial disparity gap — Mortality vs Survival prediction\n'
             'Does flipping the outcome change the bias?')
ax.legend(fontsize=10)
ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.savefig(EXP_DIR / 'fig_survival_gap_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_survival_gap_comparison.png')

In [ ]:
# Figure 3: Race feature vs No race feature
models_plot = [m for m in ['XGBoost','Deep NN','LSTM','Transformer']
               if m in oof_race_feat and m in oof_baseline]
x     = np.arange(len(KEEP_RACES))
width = 0.2

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, (probs_dict, title) in [
    (axes[0], (oof_baseline,   'No race feature (baseline)')),
    (axes[1], (oof_race_feat,  'With race feature (explicit)')),
]:
    for offset, m in enumerate(models_plot):
        if m not in probs_dict: continue
        vals = [average_precision_score(
                    y_mort[race_arr==g], probs_dict[m][race_arr==g])
                for g in KEEP_RACES]
        ax.bar(x + offset*width, vals, width, label=m,
               color=list(COLORS.values())[offset],
               edgecolor='white', alpha=0.85)
    ax.set_xticks(x + width*1.5)
    ax.set_xticklabels(KEEP_RACES)
    ax.set_ylabel('AUPRC')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.set_ylim(0.3, 0.5)

plt.suptitle('Effect of adding race as explicit feature\n'
             'Does it help or hurt fairness?', fontsize=13)
plt.tight_layout()
plt.savefig(EXP_DIR / 'fig_race_feature_auprc.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_race_feature_auprc.png')

In [ ]:
# Figure 4: Three-way fairness comparison 
adv_path = OUT_DIR / 'adversarial' / 'oof_mort_lam3.0.npy'
has_adv  = adv_path.exists()

metric_names = ['equal_opportunity', 'equalized_odds', 'ber_gap']
metric_labels = ['Equal Opportunity\n(TPR gap)', 'Equalized Odds\ngap', 'BER gap']

# Use LSTM for the three-way comparison
conditions = {'Baseline\n(no race)': oof_baseline.get('LSTM')}
conditions['With race\nfeature']    = oof_race_feat.get('LSTM')
if has_adv:
    adv_probs = np.load(adv_path)
    # align to current cohort
    adv_probs_aligned = adv_probs[kept_idx][tab_mask] \
        if len(adv_probs) > len(kept_idx) else adv_probs
    if len(adv_probs_aligned) == len(y_mort):
        conditions['Adversarial\n(λ=3)'] = adv_probs_aligned

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

cond_names = list(conditions.keys())
cond_colors = ['#7f8c8d', '#e74c3c', '#3498db']

for ax, mkey, mlabel in zip(axes, metric_names, metric_labels):
    vals = []
    for cname, probs in conditions.items():
        if probs is None:
            vals.append(np.nan)
            continue
        _, gaps = compute_fairness_metrics(y_mort, probs, race_arr)
        vals.append(gaps[mkey])
    bars = ax.bar(cond_names, vals,
                  color=cond_colors[:len(cond_names)],
                  edgecolor='white', alpha=0.85)
    for bar, val in zip(bars, vals):
        if not np.isnan(val):
            ax.text(bar.get_x()+bar.get_width()/2, val+0.001,
                    f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')
    ax.set_ylabel('Gap (Black vs White)')
    ax.set_title(f'{mlabel}\nLower = more equitable')
    ax.axhline(0, color='black', linestyle='--', linewidth=0.8)

plt.suptitle('Three-way fairness comparison — LSTM\n'
             'Baseline vs Race feature vs Adversarial debiasing',
             fontsize=13)
plt.tight_layout()
plt.savefig(EXP_DIR / 'fig_threeway_fairness.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_threeway_fairness.png')

---
## 8. Summary table

In [ ]:
print('=' * 75)
print('FULL SUMMARY')
print('=' * 75)

print('\nExperiment 1: Survival prediction (LSTM)')
if 'LSTM' in oof_surv:
    p = oof_surv['LSTM']
    print(f'  Overall AUPRC: {average_precision_score(y_surv, p):.4f}')
    print(f'  AUROC:         {roc_auc_score(y_surv, p):.4f}')
    for g in KEEP_RACES:
        mask = race_arr == g
        ap = average_precision_score(y_surv[mask], p[mask])
        print(f'  {g}: {ap:.4f}')
    _, gaps = compute_fairness_metrics(y_surv, p, race_arr)
    print(f'  White-Black gap:     {gaps["auprc_gap"]:+.4f}')
    print(f'  Equal opportunity:   {gaps["equal_opportunity"]:.4f}')
    print(f'  Equalized odds:      {gaps["equalized_odds"]:.4f}')
    print(f'  BER gap:             {gaps["ber_gap"]:.4f}')

print('\n Experiment 2: Race as explicit feature (LSTM)')
if 'LSTM' in oof_race_feat:
    p = oof_race_feat['LSTM']
    p_base = oof_baseline.get('LSTM')
    print(f'  Overall AUPRC: {average_precision_score(y_mort, p):.4f}  '
          f'(baseline: {average_precision_score(y_mort, p_base):.4f})')
    for g in KEEP_RACES:
        mask = race_arr == g
        ap_r = average_precision_score(y_mort[mask], p[mask])
        ap_b = average_precision_score(y_mort[mask], p_base[mask])
        print(f'  {g}: {ap_r:.4f}  (baseline: {ap_b:.4f}  delta: {ap_r-ap_b:+.4f})')
    _, gaps_r = compute_fairness_metrics(y_mort, p, race_arr)
    _, gaps_b = compute_fairness_metrics(y_mort, p_base, race_arr)
    print(f'  White-Black gap:     {gaps_r["auprc_gap"]:+.4f}  '
          f'(baseline: {gaps_b["auprc_gap"]:+.4f})')
    print(f'  Equal opportunity:   {gaps_r["equal_opportunity"]:.4f}  '
          f'(baseline: {gaps_b["equal_opportunity"]:.4f})')
    print(f'  Equalized odds:      {gaps_r["equalized_odds"]:.4f}  '
          f'(baseline: {gaps_b["equalized_odds"]:.4f})')
    print(f'  BER gap:             {gaps_r["ber_gap"]:.4f}  '
          f'(baseline: {gaps_b["ber_gap"]:.4f})')

print(f'\nOutputs saved to: {EXP_DIR}')
for f in sorted(EXP_DIR.glob('*.png')):
    print(f'  {f.name}')
for f in sorted(EXP_DIR.glob('*.csv')):
    print(f'  {f.name}')